# Lesson 3 — Modeling in `discopt`

**Track:** Basic &nbsp;|&nbsp; **Estimated time:** 90 minutes

## Learning objectives

1. Use the `Model` API: variables, constraints, objective.
2. Build expressions with operator overloading and recognize the underlying DAG.
3. Use vectorized variables, indexing, and `sum()` patterns.
4. Detect modeling errors that the solver will turn into infeasibility/unboundedness.

## 1. The four-step recipe

Every `discopt` model is:

```python
m = do.Model("name")
x = m.continuous("x", shape=(n,), lb=0, ub=10)   # 1. variables
m.subject_to(...)                                # 2. constraints
m.minimize(...)                                  # 3. objective
result = m.solve()                               # 4. solve
```

Variable factories: `m.continuous(name, shape=..., lb=..., ub=...)`,
`m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`.
Aggregations like row/column sums use `dm.sum(expr, axis=k)` (not
`expr.sum()`). The expression DAG is built lazily as you compose operators.

See {cite:p}`Williams2013` for the broader modeling philosophy.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

m = do.Model("transport")
nS, nD = 3, 4
x = m.continuous("x", shape=(nS, nD), lb=0, ub=200)  # ship from supply i to demand j

supply = np.array([100, 150, 80])
demand = np.array([60, 90, 70, 110])
cost   = np.array([
    [4, 6, 9, 8],
    [5, 4, 7, 6],
    [9, 8, 3, 4],
])

# Supply: row sums <= supply   (one vectorized constraint, nS rows)
m.subject_to(dm.sum(x, axis=1) <= supply, name="supply")

# Demand: column sums >= demand (one vectorized constraint, nD rows)
m.subject_to(dm.sum(x, axis=0) >= demand, name="demand")

m.minimize(dm.sum(cost * x))
r = m.solve()
print("cost:", r.objective)
print("shipments:")
print(r.value(x))
print(m.summary())


## 2. Expression DAG

Every operator (`+`, `*`, `**`, `<=`, etc.) returns an Expression node, not a number. At solve time `discopt` compiles the DAG once — into the Rust model representation used by presolve and the relaxations, and into an automatic-differentiation tape for the NLP solver. Two practical implications:

- **Reuse** subexpressions: the same Python expression object used twice is one shared node in the DAG: `e = x[0]*x[1]; m.subject_to(e <= 10); m.minimize(e + x[2])`.
- **Avoid Python-level loops** for big models; prefer vectorized expressions or `sum()` over generators.

## 3. Common modeling tricks

| Want | How |
|------|-----|
| Either-or constraint | Big-M with binary indicator (Lesson 5) |
| Absolute value $|y - a|$ | Auxiliary $u \ge y - a$, $u \ge a - y$ (in min objective only) |
| Max of expressions $\max(e_1, e_2)$ | Aux $u \ge e_i$, minimize $u$ |
| Piecewise linear $f(x)$ | SOS2 or convex combination formulation |
| Logical $z = x \land y$ | $z \le x$, $z \le y$, $z \ge x + y - 1$ |

See {cite:p}`Williams2013,Wolsey1998` for the full catalogue.

## 4. Bounds matter

Tight bounds on variables are crucial:

- They **make presolve effective** {cite:p}`Savelsbergh1994,Belotti2009`.
- For nonconvex problems they **enable McCormick relaxations** {cite:p}`McCormick1976` (Lesson 16).
- Loose or missing bounds lead to long solves or unbounded LP relaxations.

If you don't know the right bound, *guess generously and shrink later*. If you omit a bound, `discopt` substitutes a huge but **finite** default box ($\pm 9.999 \times 10^{19}$) and emits a `UserWarning` naming the variables marked `[default]`. The LP route treats a `[default]` side as open, so a genuinely unbounded LP is still reported `"unbounded"`; other model classes may instead be solved over the finite box. An omitted bound on an `m.integer` variable falls back to `[0, 10^6]` (also with a warning). Declare `ub=np.inf` explicitly if you really mean "no bound", and a finite bound whenever you know one.

## 5. Diagnosing a bad model

Before blaming the solver:

1. **Look at the model.** `print(m.summary())` (or just `m` in a notebook) lists the variable, constraint and parameter counts and the objective; `m.num_variables` / `m.num_constraints` give the numbers. `m.to_lp()` returns the model as LP-format text with one line per scalar row — the quickest way to *see* what a vectorized constraint expanded into. `m.validate()` runs the same consistency checks `solve()` runs first (objective set, unique names, `lb <= ub`, ...) and raises a `ValueError` naming the problem.
2. **Solve the LP relaxation** of an MILP — if it's infeasible, your model is wrong.
3. **Ask for an IIS.** When `r.status == "infeasible"`, `m.compute_iis()` returns an *irreducible infeasible subsystem*: a minimal set of constraints and bounds that conflict. Usually it is two or three lines out of hundreds.
4. **Slacken** suspect constraints with a penalty term — see which ones are tight in your "wrong" answer.

`r.explain()` gives a one-line plain-English summary of any result. To share a model, `m.save("model.dopt")` / `dm.load("model.dopt")` round-trip it exactly (names included); `m.to_mps()`, `m.to_lp()`, `m.to_nl()` and `m.to_gams()` export it for other solvers.

These tricks save days of debugging.

In [ ]:
# A deliberately broken transport model: demand now exceeds total supply.
bad = do.Model("transport_bad")
xb = bad.continuous("x", shape=(nS, nD), lb=0, ub=200)
bad.subject_to(dm.sum(xb, axis=1) <= supply, name="supply")
bad.subject_to(dm.sum(xb, axis=0) >= demand + 50, name="demand")
bad.minimize(dm.sum(cost * xb))
bad.validate()                      # structurally fine -- no exception
rb = bad.solve()
print("status:", rb.status, "|", rb.explain())
iis = bad.compute_iis()
print(iis.summary())